# 👁️ AI Smart Camera: Real-Time Object Detection & Counting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tahadeab-ai/ai-smart-camera/blob/main/notebooks/01_ai_smart_camera_colab.ipynb)
[![GitHub License](https://img.shields.io/badge/License-MIT-blue.svg)](https://github.com/tahadeab-ai/ai-smart-camera/blob/main/LICENSE)
[![Model-YOLOv8](https://img.shields.io/badge/Model-YOLOv8%20%7C%20YOLO11-orange.svg)](https://docs.ultralytics.com/)
[![Supervision](https://img.shields.io/badge/Vision-Supervision-green.svg)](https://supervision.roboflow.com/)

Welcome to the official **Google Colab Interactive Execution Environment** for the **AI Smart Camera** project.

This notebook guides you through hardware inspection, automated repository installation, checksum-verified model loading, computer vision inference, live mobile browser camera streaming, benchmarking, automated unit testing, and GitHub release management.

---

## 1. Environment & Hardware Inspection
Inspect the Colab runtime environment, including Python version, memory, and CUDA GPU acceleration (with automatic CPU fallback).

In [ ]:
import sys
import os
import platform
import psutil
import torch

print("=" * 60)
print("AI SMART CAMERA - HARDWARE & ENVIRONMENT AUDIT")
print("=" * 60)
print(f"Python Version   : {platform.python_version()} ({platform.python_implementation()})")
print(f"Operating System : {platform.system()} {platform.release()}")
print(f"CPU Logical Cores: {psutil.cpu_count(logical=True)}")
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"System Memory    : {ram_gb:.2f} GB RAM")
print(f"PyTorch Version  : {torch.__version__}")

cuda_available = torch.cuda.is_available()
print(f"CUDA Accelerated : {cuda_available}")
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Model        : {gpu_name} ({gpu_mem:.2f} GB VRAM)")
    selected_device = "cuda"
else:
    print("GPU Model        : None detected (Using high-performance CPU inference)")
    selected_device = "cpu"

print(f"Active Device    : {selected_device.upper()}")
print("=" * 60)

## 2. Repository Setup & Dependency Installation
Clone the official `ai-smart-camera` GitHub repository and install pinned dependencies.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/tahadeab-ai/ai-smart-camera.git"
PROJECT_DIR = "/content/ai-smart-camera"

if not os.path.exists(PROJECT_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repository already cloned. Pulling latest changes...")
    !cd {PROJECT_DIR} && git pull

%cd {PROJECT_DIR}
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("Installing dependencies from requirements-dev.txt...")
!pip install -q -r requirements-dev.txt

import ultralytics
import supervision as sv
import gradio as gr
import cv2
import PIL

print("\nVerified Installed Core Packages:")
print(f"- Ultralytics : {ultralytics.__version__}")
print(f"- Supervision : {sv.__version__}")
print(f"- Gradio      : {gr.__version__}")
print(f"- OpenCV      : {cv2.__version__}")
print(f"- Pillow      : {PIL.__version__}")

## 3. Model Manifest & Checksum-Verified Weight Loading
Load metadata from `models/manifest.yaml` and verify model integrity with SHA-256.

In [ ]:
from app.core.model_loader import ModelLoader

loader = ModelLoader(manifest_path="models/manifest.yaml", cache_dir="models/weights")
MODEL_NAME = "yolov8n"

print(f"Fetching and verifying model: {MODEL_NAME}...")
model, metadata = loader.load(model_name=MODEL_NAME, device=selected_device)

print("\n--- Model Metadata ---")
for key, val in metadata.items():
    if key != "classes":
        print(f"{key:20}: {val}")
print(f"Supported Categories : {metadata.get('classes_count', 80)} classes")

## 4. Basic Inference with Roboflow Supervision
Test object detection and annotation on a sample image.

In [ ]:
import urllib.request
import numpy as np
from PIL import Image
from IPython.display import display
from app.services.camera_service import CameraService

sample_url = "https://raw.githubusercontent.com/roboflow/supervision/develop/assets/examples/traffic-analysis.png"
sample_path = "sample_test.png"
try:
    urllib.request.urlretrieve(sample_url, sample_path)
    test_img = np.array(Image.open(sample_path).convert("RGB"))
except Exception:
    # Fallback to synthetic image
    test_img = np.zeros((480, 640, 3), dtype=np.uint8)
    cv2.putText(test_img, "Traffic Test", (50, 240), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

service = CameraService(model_name=MODEL_NAME, device=selected_device)
orig, annotated, stats, latency = service.process_image(test_img, confidence_threshold=0.25)

print(f"Inference Latency : {latency:.2f} ms")
print(f"Total Detected    : {stats['total_count']}")
print(f"Class Breakdown   : {stats['class_counts']}")

# Display annotated output
display(Image.fromarray(annotated))

## 5. Performance Evaluation & Benchmarking
Measure actual inference latency over repeated iterations to compute mean, P50, and P95 latency.

In [ ]:
import time

WARMUP_RUNS = 5
BENCHMARK_RUNS = 25
latencies = []

print(f"Running {WARMUP_RUNS} warm-up iterations...")
for _ in range(WARMUP_RUNS):
    service.process_image(test_img, confidence_threshold=0.25)

print(f"Measuring latency over {BENCHMARK_RUNS} benchmark iterations...")
for i in range(BENCHMARK_RUNS):
    _, _, _, lat = service.process_image(test_img, confidence_threshold=0.25)
    latencies.append(lat)

lat_arr = np.array(latencies)
mean_lat = np.mean(lat_arr)
p50_lat = np.percentile(lat_arr, 50)
p95_lat = np.percentile(lat_arr, 95)
fps_est = 1000.0 / mean_lat if mean_lat > 0 else 0

print("=" * 50)
print("BENCHMARK RESULTS")
print("=" * 50)
print(f"Device                : {selected_device.upper()}")
print(f"Model Architecture    : {MODEL_NAME}")
print(f"Image Resolution      : {test_img.shape[1]}x{test_img.shape[0]}")
print(f"Mean Inference Latency: {mean_lat:.2f} ms")
print(f"Median (P50) Latency  : {p50_lat:.2f} ms")
print(f"P95 Latency           : {p95_lat:.2f} ms")
print(f"Estimated Max FPS     : {fps_est:.1f} FPS")
print("=" * 50)

## 6. Automated Unit Tests Execution
Execute `pytest` on the test suite to verify vision modules, input validation, and model loading logic.

In [ ]:
!pytest -v tests/

## 7. Launch Live Mobile-Ready Web Application
Launch the Gradio user interface with public HTTPS tunneling (`share=True`).

> **Mobile Access Tip:** Open the generated `https://xxxx.gradio.live` link directly in Safari or Chrome on your smartphone. Allow camera permissions when prompted to stream your mobile camera in real-time.

In [ ]:
from app.ui.gradio_app import create_gradio_app

demo = create_gradio_app(camera_service=service)
demo.queue()
app_url, share_url, _ = demo.launch(
    share=True,
    inline=True,
    show_error=True,
)

print("\n" + "=" * 60)
print(f"PUBLIC MOBILE & WEBCAM ACCESS URL:\n{share_url}")
print("=" * 60)

## 8. GitHub Release & Artifact Publishing (Controlled Workflow)
Publish release metadata, checksums, and release notes to GitHub using Colab Secrets.

In [ ]:
import os
from google.colab import userdata

try:
    github_token = userdata.get('GITHUB_TOKEN')
    print("✅ GITHUB_TOKEN retrieved successfully from Colab Secrets.")
    os.environ["GITHUB_TOKEN"] = github_token
    !gh auth status
except Exception as e:
    print("ℹ️ GITHUB_TOKEN not configured in Colab Secrets.")
    print("To publish releases from Colab, add GITHUB_TOKEN with 'repo' scope to Secrets (key icon in sidebar).")